# 1 Importing Packages

How the *MESSAGE_CA* class works: the class will contain all the functions necessary to run the MESSAGEix Canada model. All that you need to run a model is to:

1. Create your scenario folder under `model/scenarios` with its own `config.toml` file

2. Declare the class reading that scenario, for example: `platform, msg, msg_folder = MESSAGE_CA(scenario_name='baseline')`

In [1]:
import os
import sys


In [3]:
from model.Model import model_MSG 
#from model.Plotter import plotter_MSG


# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

In [7]:
model_MSG.clone_scenario

NameError: name 'model_MSG' is not defined

In [2]:
# Sys and OS Operations

os.chdir("../../")
from pathlib import Path

print(Path.cwd())

/Users/omarattia/Repos/message-ix


# Scenario Creation and Set-up

Setting up the MESSAGEix scenario using the scenario configuration file and an input file for technology options in technology.yaml

In [58]:
# Set-upmsg = model_MSG(scenario_name=scenario, config=None)
msg = model_MSG(scenario_name='baseline', config=None)
platform, msg_scenario, msg_folder = msg.setup_scenario()


Model Logs: Setting up scenario

  1 elements not found: ['import']
from gen_data_cement()
adjust regression parameters according to mode: normal
before adjustment: [  514.21381146 -3017.70274274]
after adjustment: [  514.21381146 -3017.70274274]
Updating efficiencies for technologies: ['coal_bal']
Updating efficiencies for technologies: ['coal_t_d']
Updating relation_activity for technology: gas_t_d,  relation CO2_cc.
Updating relation_activity for technology: gas_t_d_ch4,  relation CO2_cc.
Skipping technology: LNG_trd; efficiency is 100%.
Adding relation_activity for technology: gas_exp_afr,  relation: CO2_trade.
Adding relation_activity for technology: gas_exp_cpa,  relation: CO2_trade.
Adding relation_activity for technology: gas_exp_eeu,  relation: CO2_trade.
Adding relation_activity for technology: gas_exp_nam,  relation: CO2_trade.
Adding relation_activity for technology: gas_exp_pao,  relation: CO2_trade.
Adding relation_activity for technology: gas_exp_sas,  relation: CO2_tra

[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered IamDataFrame is empty!
[WARNING] 11:51:41 - pyam.core: Filtered

Values within acceptable deviation: 55
Values outside acceptable deviation: 716


In [61]:
msg_scenario.has_solution()

False

In [57]:
df = msg.ctx.source_scenario['scenario'].par("output")
print(list(df.columns))
# Filter
filter_condition = (
    (df['commodity'] == 'freshwater_supply') &
    (df['node_loc'] == 'Ontario')
)
#msg.ctx.source_scenario['scenario'].remove_solution()
msg_scenario.check_out()
msg_scenario.remove_par('output', df)

df.loc[filter_condition, 'value'] *= 1/1.4

msg.ctx.source_scenario['scenario'] #.set_par("output", df)

msg_scenario.add_par('output', df)


msg_scenario.commit("added parameters/sets")

msg_scenario.check_out()

print(list(df.loc[filter_condition, 'value']))


['node_loc', 'technology', 'year_vtg', 'year_act', 'mode', 'node_dest', 'commodity', 'level', 'time', 'time_dest', 'value', 'unit']


RuntimeError: unhandled Java exception: This Scenario was not yet saved to the database - do a commit first!

In [54]:
msg_scenario.commit("g")

RuntimeError: unhandled Java exception: There was a problem writing data to the IXMP database - no changes were saved!
Cannot create keys in DB!

In [45]:
df = msg.ctx.source_scenario['scenario'].par("output")
print(list(df.loc[filter_condition, 'value']))
#msg_scenario.remove_solution()

# msg.ctx.source_scenario['scenario'].check_out()
msg_scenario.commit("added parameters/sets")

msg.ctx.source_scenario['scenario'].solve(
    solve_options={"write_model_to_file":"true", "writelp":"true", "write_model_file":"true", "lpmethod":2}
)

[1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4, 1.4]


RuntimeError: unhandled Java exception: There was a problem writing data to the IXMP database - no changes were saved!
Cannot create keys in DB!

In [18]:
fix_cost_data = msg_scenario.par('fix_cost')
filtered_data = fix_cost_data[(fix_cost_data['technology'] == 'gas_ppl') & 
                              (fix_cost_data['year_act'].isin([2020, 2025, 2040])) & (fix_cost_data['year_vtg'].isin([2020, 2025, 2040]))]


print(filtered_data)


            node_loc technology  year_vtg  year_act  value    unit
24491       Manitoba    gas_ppl      2020      2020  20.53  USD/kW
24492       Manitoba    gas_ppl      2020      2025  19.40  USD/kW
24495       Manitoba    gas_ppl      2020      2040  16.83  USD/kW
24497       Manitoba    gas_ppl      2025      2025  19.40  USD/kW
24500       Manitoba    gas_ppl      2025      2040  16.83  USD/kW
...              ...        ...       ...       ...    ...     ...
143741  Saskatchewan    gas_ppl      2025      2040  16.83  USD/kW
143756  Saskatchewan    gas_ppl      2040      2040  16.83  USD/kW
143771  Saskatchewan    gas_ppl      2020      2020  20.53  USD/kW
143772  Saskatchewan    gas_ppl      2020      2025  19.40  USD/kW
143775  Saskatchewan    gas_ppl      2020      2040  16.83  USD/kW

[78 rows x 6 columns]


In [ ]:
import pandas as pd
import pyam as pyam
import message_ix as message_ix

In [ ]:
node_commodity=[("Alberta", "primary", "gas", 2020)]


In [ ]:

for node, lvl, com, yr in node_commodity:
    # Demand
    dem = msg_scenario.par(
        "demand", {"commodity": com, "node": node, "level": lvl, "year": yr}
    )
    dem = float(dem["value"])
    # Activity
    out = msg_scenario.par("output", {"commodity": com, "node_loc": node, "year_act": yr})
    tec_list = list(set(out["technology"]))
    act = msg_scenario.par(
        "historical_activity", {"technology": tec_list, "node_loc": node, "year_act": yr}
    )["value"].sum()

In [ ]:
#Adding hydrogen storage with seasonality

msg_scenario.add_set("storage_tec", "h2_stor_geo")

msg_scenario.add_set("level_storage", "storage")


time_steps = ["winter", "spring", "summer", "fall"]
msg_scenario.add_set("time", time_steps)

time_level = "season"
msg_scenario.add_set("lvl_temporal", time_level)

for t in time_steps:
    msg_scenario.add_set("map_temporal_hierarchy", [time_level, t, "year"])

for t in time_steps:
    msg_scenario.add_par("duration_time", [t], 0.25, "-")

def yearly_to_season(scen, parameter, data, tech_list, filters=None):
    if filters:
        old = msg_scenario.par(parameter, filters)
    else:
        old = msg_scenario.par(parameter)
    
    # Filter for specified technologies
    old = old[old.technology.isin(tech_list)]
    
    msg_scenario.remove_par(parameter, old)

    time_idx = [x for x in scen.idx_names(parameter) if "time" in x]
    for tech in tech_list:
        if tech in data[parameter]:
            tech_data = old[old.technology == tech].copy()
            for h in data[parameter][tech].keys():
                new = tech_data.copy()
                for time in time_idx:
                    new[time] = h
                new["value"] = data[parameter][tech][h] * new["value"]
                msg_scenario.add_par(parameter, new)

tech_list = ["h2_stor_geo", "h2_stor_geo_charge", "h2_stor_geo_discharge"]

seasonal_data = {
    "input": {tech: {'winter': 1, 'spring': 1, 'summer': 1, 'fall': 1} for tech in tech_list},
    "output": {tech: {'winter': 1, 'spring': 1, 'summer': 1, 'fall': 1} for tech in tech_list},
    "capacity_factor": {tech: {'winter': 1, 'spring': 1, 'summer': 1, 'fall': 1} for tech in tech_list},
    "var_cost": {tech: {'winter': 1, 'spring': 1, 'summer': 1, 'fall': 1} for tech in tech_list}
}

# For input parameter
yearly_to_season(msg_scenario, "input", seasonal_data, tech_list)

# For output parameter
yearly_to_season(msg_scenario, "output", seasonal_data, tech_list)

# For cf parameter
yearly_to_season(msg_scenario, "capacity_factor", seasonal_data, tech_list)

# For var cost parameter
yearly_to_season(msg_scenario, "var_cost", seasonal_data, tech_list)\

for province in provinces:
    msg_scenario.add_set("map_tec_storage", [province, "h2_stor_geo_charge", "charge", "h2_stor_geo", "M1", "storage", "hydrogen", "season"])
    msg_scenario.add_set("map_tec_storage", [province, "h2_stor_geo_discharge", "discharge", "h2_stor_geo", "M1", "storage", "hydrogen", "season"])

for year in years:
    for province in provinces:
        for step in time_steps:
            msg_scenario.add_par("storage_self_discharge", 
                 {"node": [province], 
                  "technology": ["h2_stor_geo"], 
                  "commodity": ["hydrogen"], 
                  "level": ["storage"],
                  "mode": ["M1"],
                  "year": [year],
                  "time": step, #slice
                  "value": [0.001]},  # Include the value in the dictionary
                 unit="-")

for province in provinces: 
    for slep in time_steps:  
        msg_scenario.add_par('storage_initial', 
                    {
                        'node': province,
                        'technology': "h2_stor_geo",
                        'year': 2020,
                        'mode': "M1",
                        'commodity': ["hydrogen"],
                        'level': ["storage"],
                        'time': "winter",
                        'value': 0.05, #constrain this
                        'unit': '-',
                    })

In [ ]:
msg_scenario.add_set("mode", "charge")
msg_scenario.add_set("mode", "discharge")
msg_scenario.add_set('level', 'storage')


In [ ]:
msg_scenario.remove_solution()

In [5]:
msg_scenario.check_out()

In [ ]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2020, 2061, 5))

In [ ]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2020, 2061, 5))

In [6]:
from message_ix import make_df

In [7]:
#Trade Hub

provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2020, 2061, 5))

# Adding new set elements for the hub
msg_scenario.add_set("node", "Trade_hub")
msg_scenario.add_set("level", "trade")

# Adding new electricity import and export technologies
msg_scenario.add_set("technology", ["lh2_imp", "lh2_exp"])

#do export investment cost of 450

#model_years = list(msg_scenario.set("year"))
for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            for province in provinces: 
                base_input = {
                    "node_loc": [province], 
                    "node_origin": ["Trade_hub"],
                    "technology": ["lh2_imp"],
                    "commodity": ["hydrogen"],
                    "level": ["trade"],
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": ["M1"],
                    "time": ["year"],
                    "time_origin": ["year"],
                    "value": [1],
                    "unit": ["-"],
                }
                msg_scenario.add_par("input", base_input)

for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            for province in provinces:    
                base_output = {
                    "node_loc": [province],
                    "node_dest": [province],
                    "technology": ["lh2_imp"],
                    "commodity": ["hydrogen"],
                    "level": ["secondary"],
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": ["M1"],
                    "time": ["year"],
                    "time_dest": ["year"],
                    "value": [1],
                    "unit": ["-"],
                }
                msg_scenario.add_par("output", base_output)

for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            for province in provinces:    
                cp = {
                    "node_loc": [province],
                    "technology": ["lh2_exp"],
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": ["M1"],
                    "time": ["year"],
                    "value": [1],
                    "unit": ["-"],
                }
                msg_scenario.add_par ("capacity_factor", cp)

for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            for province in provinces:   
                exp_input = {
                    "node_loc": [province],
                    "node_origin": [province],
                    "technology": ["lh2_exp"],
                    "commodity": ["hydrogen"],
                    "level": ["secondary"],
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": ["M1"],
                    "time": ["year"],
                    "time_origin": ["year"],
                    "value": [1.05],
                    "unit": ["-"],
                }
                msg_scenario.add_par("input", exp_input)


for year_act in years:
    for year_vtg in years:
        if year_vtg <= year_act <= year_vtg + 40:
            for province in provinces:   
                exp_output = {
                    "node_loc": [province],
                    "node_dest": ["Trade_hub"],
                    "technology": ["lh2_exp"],
                    "commodity": ["hydrogen"],
                    "level": ["trade"],
                    "year_vtg": [year_vtg],
                    "year_act": [year_act],
                    "mode": ["M1"],
                    "time": ["year"],
                    "time_dest": ["year"],
                    "value": [1],
                    "unit": ["-"],
                }
                msg_scenario.add_par("output", exp_output)


tecs=['lh2_imp','lh2_exp']

for tec in tecs:
    for year_vtg in years:    
        for province in provinces:    
            inv_cost = {
                "node_loc": [province],
                "technology": [tec], 
                "year_vtg": [year_vtg],
                "value": [225],
                "unit": ["USD/kW"],
            }
            msg_scenario.add_par("inv_cost", inv_cost)

for tec in tecs:
    for year_vtg in years:    
        for province in provinces: 
            tec_lifetime = {
                "node_loc": [province],
                "technology": [tec], 
                "year_vtg": [year_vtg],
                "value": [40],
                "unit": ["year"],
            }
            msg_scenario.add_par("technical_lifetime", tec_lifetime)



In [ ]:
filtered_data = msg_scenario.par("technical_lifetime", filters={"technology": "lh2_imp"})
print(filtered_data)

In [ ]:
#This was used to fix the add_technology function

dfs = []

for year in years:
    emission_factor_df = message_ix.make_df(
        "relation_activity",
        node_rel=provinces,
        node_loc=provinces,
        technology=['h2_smr'] * len(provinces),
        mode=['M1'] * len(provinces),
        value=[0.5] * len(provinces),
        relation=['CO2_cc'] * len(provinces),
        unit=["-"] * len(provinces),
        year_rel=[year] * len(provinces),
        year_act=[year] * len(provinces),
    )
    dfs.append(emission_factor_df)

# Concatenate all DataFrames
combined_df = pd.concat(dfs, ignore_index=True)

# Add the combined DataFrame to the scenario
msg_scenario.add_par('relation_activity', combined_df)



In [ ]:
relation_activity_data = msg_scenario.par('relation_activity', filters={'technology': ['h2_pyrolysis'], 'relation': ['CO2_cc']})
print(relation_activity_data)


In [ ]:
'''provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

shares = ['share_h2_fc_i', 'share_h2_i', 'share_h2_fc_RC', 'share_h2_rc']

years = list(range(2025, 2061, 5))

# Get the entire share_commodity_lo parameter once
share_df = msg_scenario.par('share_commodity_lo')

# Rename columns to match expected dimensions
share_df = share_df.rename(columns={'node_share': 'node', 'year_act': 'year'})

# Print information about the DataFrame
print("\nDataFrame Info:")
share_df.info()

print("\nUnique values in 'shares' column:")
print(share_df['shares'].unique())

print("\nUnique values in 'node' column:")
print(share_df['node'].unique())

print("\nUnique values in 'year' column:")
print(share_df['year'].unique())

print("\nUnique values in 'time' column:")
print(share_df['time'].unique())

# Create a mask for the rows we want to modify
mask = (share_df['shares'].isin(shares)) & \
       (share_df['node'].isin(provinces)) & \
       (share_df['year'].isin(years)) & \
       (share_df['time'] == 'year')

# Print the number of True values in each part of the mask
print("\nNumber of True values in each part of the mask:")
print("shares mask:", share_df['shares'].isin(shares).sum())
print("node mask:", share_df['node'].isin(provinces).sum())
print("year mask:", share_df['year'].isin(years).sum())
print("time mask:", (share_df['time'] == 'year').sum())

# Multiply the values for the masked rows by 1.2
share_df.loc[mask, 'value'] *= 0.9

# Print summary to verify changes
print("\nSummary of modifications:")
print(share_df[mask]['value'].describe())

# Print the number of modified rows
print(f"\nNumber of rows modified: {mask.sum()}")

# Print unique values in 'shares' column to verify which shares were affected
print("\nShares affected:")
print(share_df[mask]['shares'].unique())

# Remove the existing parameter
msg_scenario.remove_par('share_commodity_lo')

msg_scenario.init_par('share_commodity_lo', ['shares', 'node', 'year', 'time'])

share_df = share_df.rename(columns={'node': 'node_share', 'year': 'year_act'})

# Add the modified parameter back to the scenario
msg_scenario.add_par('share_commodity_lo', share_df)'''

In [ ]:


'''years = list(range(2025, 2041, 5))
technology = "h2_atr_ccs"
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Loop through provinces and years to remove the 'input' parameter
for province in provinces:
    for year_act in years:
        for year_vtg in years:
            if year_vtg <= year_act <= year_vtg + 10:
                msg_scenario.remove_par('input', 
                    {
                        'node_loc': province,
                        'technology': technology,
                        'year_vtg': year_vtg,
                        'year_act': year_act,
                        'mode': "M1",
                        'node_origin': province,
                        'commodity': ["electr"],
                        'level': ["primary"],
                        'time': "year",
                        'time_origin': "year"
                    }
                )'''

In [ ]:
'''years = list(range(2020, 2041, 5))
technology = "h2_smr_ccs"
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Loop through provinces and years to remove the 'input' parameter
for province in provinces:
    for year_act in years:
        for year_vtg in years:
            if year_vtg <= year_act <= year_vtg + 25:
                msg_scenario.remove_par('emission_factor', 
                    {
                        'node_loc': province,
                        'technology': technology,
                        'year_vtg': year_vtg,
                        'year_act': year_act,
                        'mode': "M1",
                        'emission': ["CO2_Emission"],
                    }
                )'''

In [ ]:
'''import pandas as pd

# Define the provinces and the years
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2030, 2061, 5))  # 2030, ..., 2060
years2 = list(range(2040, 2061, 5)) 
technologies = ['h2_atr_ccs', 'meth_h2']
technology2 = 'h2_gas_fc_trp'
value = 0.015
value2= 0.12162

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('initial_activity_up', 
                        [province, technology, year, "year"], 
                        value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years2:
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology2, year, "year"], 
                    value2, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed       
    for technology in technologies:      
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology, "2020", "year"], 
                    0.006, 'GWa')
    for technology in technologies:
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology, "2025", "year"], 
                    0.0105, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2025", "year"], 
                0.10815, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2030", "year"], 
                0.11295, 'GWa')

    msg_scenario.add_par('initial_activity_up', 
                [province, technology2, "2035", "year"], 
                0.1173, 'GWa')'''



In [ ]:
value = 0.08
value2= 0.075
years = list(range(2020, 2061, 5)) 

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('growth_activity_up', 
                        [province, technology, year, "year"], 
                        value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
        
        msg_scenario.add_par('growth_activity_up', 
                    [province, technology2, year, "year"], 
                    value2, 'GWa')

In [ ]:
value = 0.006
#h2_atr
technologies= ['h2_atr_ccs', 'meth_h2', 'h2_atr'] #meed to do for both w and without ccs

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('initial_new_capacity_up', 
                    [province, technology, year], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
            


In [ ]:
value = 0.05 #meed to do for both w and without ccs
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('growth_new_capacity_up', 
                    [province, technology, year], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed'''

In [ ]:
technology = "h2_pyrolysis"
years = list(range(2020, 2061, 5)) 

for year in years:
    for province in provinces:
        msg_scenario.add_par('growth_new_capacity_up', 
                        [province, technology, year], 
                        0.05, 'GWa')
        
        msg_scenario.add_par('initial_new_capacity_up', 
                        [province, technology, year], 
                        0.006, 'GWa')
        
        msg_scenario.add_par('relation_activity', 
                        ["h2_exp_limit", province, year, province, technology, year, "M1"], 
                        1, 'GWa/a')
        
    



In [ ]:
msg_scenario.add_set("cat_tec", ["transformation_fossil", "h2_atr_ccs"])

msg_scenario.add_set("cat_tec", ["powerplant", "h2_atr_ccs"])

In [ ]:
'''msg_scenario.add_set("cat_tec", ["h2_trp", "h2_gas_fc_trp"])

msg_scenario.add_set("cat_tec", ["transport", "h2_gas_fc_trp"])

# Optionally, you can check if the addition was successful
cat_tec = msg_scenario.set("cat_tec")
print(cat_tec)'''

In [ ]:
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']
years = list(range(2020, 2061, 5)) 
value = -0.62319
value2= 1

for province in provinces:
    for year in years:
      msg_scenario.add_par('relation_activity', 
                    ["h2_scrub_limit", province, year, province, "h2_atr_ccs", year, "M1"], 
                    value, 'GWa/a') 
      
      msg_scenario.add_par('relation_activity', 
                    ["h2_exp_limit", province, year, province, "h2_atr_ccs", year, "M1"], 
                    value2, 'GWa/a')
      msg_scenario.add_par('relation_activity', 
                    ["h2_exp_limit", province, year, province, "h2_atr", year, "M1"], 
                    value2, 'GWa/a') 
      

In [8]:
msg_scenario.commit("added parameters/sets")

In [6]:
msg_scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"})

In [ ]:

"""
Apply the Output-Based Pricing System (OBPS) policy for the Baseline Canada Scenario
to the given MESSAGEix scenario.

This function modifies the scenario's fixed costs based on emissions intensity and carbon pricing.

Args:
    scenario (ixmp.Scenario): The MESSAGEix scenario to apply the OBPS policy to.
    data_file_path (str): Path to the Excel file containing necessary data.
"""

'''
# Read the required data
data_relation = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='emission_factor')

provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
            'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
            'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Define conditions for filtering the data
conditions_2 = (
    data_relation['technology'].isin([
        'oil_extr_1', 'oil_extr_2', 'oil_extr_3', 'oil_extr_4', 'oil_extr_5', 'oil_extr_6',
        'cement_CO2', 'coal_extr', 'gas_extr_1', 'gas_extr_2', 'gas_extr_3', 'gas_extr_4', 
        'gas_extr_5', 'gas_extr_6', 'h2_smr_ccs', 'h2_smr', 'h2_atr', 'h2_atr_ccs'
    ]) &
    (data_relation['emission'] == 'CO2_cc') &
    (data_relation['node_loc'].isin(provinces))
)

# Filter the data based on the conditions
result = data_relation[conditions_2][['technology', 'value', 'node_loc', 'year_act']]

# Initialize columns for calculations
result['Emission Intensity Reduction_baseline'] = 0.0

# Calculate Emission Intensity Reduction_baseline
for index, row in result.iterrows():
    if 2020 < row['year_act'] < 2055:
        previous_year_value = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == row['year_act'] - 5), 
            'Emission Intensity Reduction_baseline'
        ].values
        if len(previous_year_value) > 0:
            result.loc[index, 'Emission Intensity Reduction_baseline'] += previous_year_value[0] + 3.33/100

# Calculate Emission intensity standard_Baseline Canada Scenario
result['Emission intensity standard_Baseline Canada Scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] > 2020:
        value_cal = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == 2015), 
            'value'
        ]
        if not value_cal.empty:
            result.at[index, 'Emission intensity standard_Baseline Canada Scenario'] = row['value'] - row['Emission Intensity Reduction_baseline'] * value_cal.values[0]
        else:
            value_cal = result.loc[
                (result['technology'] == row['technology']) & 
                (result['year_act'] == 2020), 
                'value'
            ]
            result.at[index, 'Emission intensity standard_Baseline Canada Scenario'] = row['value'] - row['Emission Intensity Reduction_baseline'] * value_cal.values[0]

# Calculate Payable emission intensity_Baseline scenario
result['Payable emission intensity_Baseline scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] > 2020:
        result.loc[index, 'Payable emission intensity_Baseline scenario'] = row['value'] - row['Emission intensity standard_Baseline Canada Scenario']

# Calculate carbon cost_Baseline Canada scenario
rate = 0.73
result['carbon cost_Baseline Canada scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] == 2025:
        result.loc[index, 'carbon cost_Baseline Canada scenario'] = row['Payable emission intensity_Baseline scenario'] * rate * 125
    elif row['year_act'] > 2025:
        result.loc[index, 'carbon cost_Baseline Canada scenario'] = row['Payable emission intensity_Baseline scenario'] * rate * 170

# Adjust for years 2055 and 2060
for index, row in result.iterrows():
    if row['year_act'] > 2050:
        value_to_assign = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == 2050), 
            'carbon cost_Baseline Canada scenario'
        ].values
        if len(value_to_assign) > 0:
            result.at[index, 'carbon cost_Baseline Canada scenario'] = value_to_assign[0]

# Merge with fix_cost data and update the scenario's fix_cost
data_relation = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='emission_factor')
# Read the fix_cost data
data_fixcost = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='fix_cost')

merged_data = pd.merge(
    data_fixcost, 
    result[['technology', 'year_act', 'node_loc', 'carbon cost_Baseline Canada scenario']],
    left_on=['technology', 'year_act', 'node_loc'],
    right_on=['technology', 'year_act', 'node_loc'],
    how='left'
)
merged_data['value'] = merged_data['value'] + merged_data['carbon cost_Baseline Canada scenario'].fillna(0)
merged_data.drop(columns=['carbon cost_Baseline Canada scenario'], inplace=True)

# Assuming 'merged_data' contains the data you've shown

for index, row in merged_data.iterrows():
    msg_scenario.add_par('fix_cost', pd.DataFrame({
        'node_loc': [row['node_loc']],
        'technology': [row['technology']],
        'year_vtg': [row['year_vtg']],
        'year_act': [row['year_act']],
        'mode': ['M1'],  # Assuming 'standard' mode for all entries
        'unit': [row['unit']],
        'value': [row['value']]
    }, index=[0]))'''

In [ ]:
import pandas as pd

In [ ]:

df_up = msg_scenario.par('bound_activity_up')
df_up = df_up[df_up['technology'].str.contains('gas_extr|oil_extr')]
# Identifying unique node_loc and technology combinations
unique_combinations = df_up[["node_loc", "technology"]].drop_duplicates()

# Creating new rows for 2055 and 2060 by copying 2050 values
new_rows = []
for _, row in df_up[df_up["year_act"] == 2050].iterrows():
    for new_year in [2055, 2060]:
        new_row = row.copy()
        new_row["year_act"] = new_year
        new_rows.append(new_row)


df_extended_lo = pd.concat([df_up, pd.DataFrame(new_rows)], ignore_index=True)

msg_scenario.add_par('bound_activity_up', df_extended_lo)

In [ ]:
df_up['technology'].unique()

In [ ]:
df_extended_lo[df_extended_lo['year_act'] >2050]

In [5]:
msg_scenario.to_excel(r"C:\Users\brend\Documents\GitHub\message-ix - CA\model\notebooks\model_default_v22_fix_cost_updates.xlsx")


In [ ]:
output_path=r"C:/Users/brend/Documents/Github/message-ix - CA/model/scenarios/baseline/MESSAGE_baseline_obps.csv"
min_year= 2015
max_year= 2060
from_excel=False
unit_correct=True

rep = message_ix.Reporter.from_scenario(
    msg_scenario,
    units={"apply": {"var_cost": "", "inv_cost": "", "fix_cost": ""}},
)
report = rep.get("message::default")

if from_excel:
    report_df = pd.read_csv("MESSAGE_NO_example_default_report.csv")
else:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()

years = msg_scenario.set("year").tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

report_df.fillna(0, inplace=True)

report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(
    report_df.columns.to_series()["2020" : str(min_year)], axis=1
)
report_df.to_csv('test.csv')
in_flows = report_df[report_df["Variable"].str.startswith("in|")]
out_flows = report_df[report_df["Variable"].str.startswith("out|")]
all_flows = pd.concat([out_flows, in_flows])

report_df["Model"] = "MESSAGEix-Canada"
report_df["Scenario"] = "Reference"
report_df["Variable"] = report_df["Variable"].str.replace(
    "emis", "Emissions", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "total om cost|", "Total Costs|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP_NEW|new capacity|", "Capacity Addition|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP|capacity|", "Capacity|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "inv cost|", "Capital Cost|", regex=False
)
report_df["Region"] = report_df["Region"].apply(lambda x: x.split("|")[0])

report_df.columns = report_df.columns.str.strip()  # Remove any leading/trailing whitespace
report_df.rename(columns={'Model': 'model', 'Scenario': 'scenario', 'Variable': 'variable'}, inplace=True)

report_df = report_df.drop_duplicates(subset=["model", "scenario", "Region", "variable"], keep='first')


pyam_df = pyam.IamDataFrame(report_df)

variable = pyam_df.variable
pyam_df.aggregate_region(variable, append=True)
pyam_df.rename({"region": {"World": "Canada"}}, inplace=True)

pyam_df.to_csv(output_path)

print(f"Report saved to {output_path}")



In [ ]:
'''import os

output_dir = r'C:\Users\brend\Documents\Github\message-ix - CA\scenarios\baseline'

# Check if the directory exists, if not, create it
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'report.csv')

# Now save the file
pyam_df.to_csv(output_path)
print(f"Report saved to {output_path}")'''

In [ ]:
print(report_df["Region"].value_counts())

In [ ]:
report_df[report_df["variable"] =='Secondary Energy|electr|h2_smr_ccs|M1']


In [ ]:
print(report_df["variable"].value_counts())

In [ ]:
report_df.columns

In [ ]:
test = report_df['variable'].value_counts()
# print("Number of unique variables:", report_df['Variable'].nunique())


In [ ]:
#from message_ix_models.tools.costs.config import Config
#from message_ix_models.tools.costs.projections import create_cost_projections

In [ ]:
'''# Step 1: Define hydrogen-related technologies
hydrogen_tech_list = ["h2_atr", "h2_atr_ccs"]

# Step 2: Define configuration for cost projections using the 'convergence' method
cfg = Config(
    module="energy",  # Assuming hydrogen falls under energy module
    method="convergence",  # Use convergence method for cost projections
    base_year=2020,  # Base year for the cost projections
    convergence_year=2050,  # Year to which costs should converge
    node="R12",  # Set your region, e.g., R12 (or your specific region code)
    ref_region="R12_NAM"  # Set the reference region
)

# Step 3: Generate base cost projections for all energy technologies
cost_projections = create_cost_projections(cfg)

# Step 4: Extract investment and fixed cost DataFrames
inv_cost_df = cost_projections["inv_cost"]
fix_cost_df = cost_projections["fix_cost"]

# Print columns for debugging
print("Investment Cost DataFrame Columns:")
print(inv_cost_df.columns)

print("\nFixed Cost DataFrame Columns:")
print(fix_cost_df.columns)

# Step 5: Filter cost projections for hydrogen-related technologies
hydrogen_inv_cost = inv_cost_df[inv_cost_df['technology'].isin(hydrogen_tech_list)]

# Debugging: Print the filtered DataFrame to understand its structure
print("\nFiltered Hydrogen Investment Cost DataFrame:")
print(hydrogen_inv_cost.head())

# Check if 'year' column is present
if 'year' not in hydrogen_inv_cost.columns:
    print("No 'year' column found in hydrogen investment costs. Available columns are:", hydrogen_inv_cost.columns)

# Step 6: Apply the learning rate manually to the investment cost
learning_rate = 0.1  # 10% learning rate

# Use 'year' instead of 'year_act' for base year costs
base_year_costs = hydrogen_inv_cost[hydrogen_inv_cost['year'] == 2020].set_index('technology')['inv_cost']

# Apply learning rate decay to future years
for year in hydrogen_inv_cost['year'].unique():
    if year > 2020:
        for tech in hydrogen_tech_list:
            # Calculate the new cost for each technology
            if tech in base_year_costs.index:
                new_cost = base_year_costs[tech] * (1 - learning_rate) ** ((year - 2020) / 5)
                # Update the DataFrame
                hydrogen_inv_cost.loc[
                    (hydrogen_inv_cost['year'] == year) & (hydrogen_inv_cost['technology'] == tech), 'inv_cost'] = new_cost

# Step 7: Extract final investment and fixed costs
inv_cost = hydrogen_inv_cost[['technology', 'year', 'inv_cost']] if 'inv_cost' in hydrogen_inv_cost.columns else None
fix_cost = fix_cost_df[fix_cost_df['technology'].isin(hydrogen_tech_list)] if 'technology' in fix_cost_df.columns else None

# You can now apply these adjusted costs to your MESSAGEix model.

# Print adjusted costs for verification
print("\nAdjusted Investment Costs:")
print(inv_cost)'''

In [ ]:
msg_scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"})

In [ ]:
msg_plt = Plotter_MSG(msg_scenario)
nodes = msg_scenario.set('node')[2:]


In [ ]:
msg_plt.plotter(case="test",nodeloc=nodes,path="output",yr_min=2020, yr_max=2060)